# Modern Transformer Architecture - Homework

In this assignment, you'll implement key innovations in modern Transformers:
- **RMSNorm**: Simpler and faster normalization
- **RoPE**: Rotary positional embeddings for better length extrapolation
- **Simplified MHLA**: Efficient attention with reduced KV cache

**What you'll learn:**
- Why modern LLMs use these techniques
- How they improve efficiency and performance
- Trade-offs between different approaches

**Submission:** Submit this completed notebook and completed python files.

**Grading:**
- Part 1 (RMSNorm): 20 points
- Part 2 (RoPE): 30 points
- Part 3 (MHLA): 35 points
- Part 4 (Integration + Questions): 15 points
Total: 100 points

## Setup

See `runner.py`.

## Part 1: RMSNorm (20 points)

See `RMSNorm.py`.

See `runner.py`.

**Question 1.1 (5 points - Written):** Why doesn't RMSNorm need to compute the mean? In what way is "controlling scale" sufficient for gradient stability?

**Why RMSNorm doesn't need to compute the mean**: While Layer Normalization standardizes both the location and scale of the post-activation values, the standardization of location has been shown to be mathematically redundant in Pre-LN residual transformer architectures. This is primarily due to the fact that, even with asymmetric activation functions, recentering information is reduntant at every normalization step. Because the mean of the activations under this framework provides no representational capacity, ensuring the network is invariant to the location is unnecessary, allowing the entire normalization operation to be reduced strictly to the RMS rescaling of the activation values.

**In what way "controlling scale" is sufficient for gradient stability**: The gradient magnitudes, which when too large will lead to exploding gradients, and when too small will lead to vanishing gradients, are primarily influenced by the magnitude of the post-activations within the forward pass. Thus, by scaling wrt. to the RMS of the post-activations, we effectively normalize the gradient and ensure information propogates throughout the entire network.

## Part 2: Rotary Positional Embeddings (30 points)

See `RoPE.py`.

See `runner.py`.

**Question 2.1 (5 points - Written):** Why does RoPE generalize better to longer sequences than learned absolute positional embeddings?

**Why RoPE generalizes better to longer sequences than learned absolute positional embeddings**: Absolute poisitonal embeddings infeasible for long sequences due to requiring the model to store a unique embedding for each position, which when given a sequence with a larger length will not be able to extrapolate. Additionally, the model is incentivized to learn spatial relationships between absolute locational values which in the case of nlp are often arbitrary, the sequence input does have spatial charecteristics but these are better captured by realtive positional information. RoPE avoids both of these issues by encoding the position through frequencies which when applied to the attention score between two tokens that are distant, decays. RoPE is also parameter free allowing it to be evaluated at any position. This naturally allows the model to extrapolate to longer sequences as the positional information is not tied to a specific position, but rather the relative distance between tokens.

# Part 3: Simplified Multi-Head Latent Attention (35 points)

See `MHLA.py`.

See `runner.py`.

**Question 3.1 (5 points - Written):** The compression ratio is constant regardless of sequence
length. Why? What does this tell you about where the memory savings come from?

**Why the compression ratio is constant regardless of sequence length**: In MHLA, the KV cache is compressed on a strictly per token basis to a fixed size latent representation. Because both the standard uncompressed memory footprint and the compressed latent footprint scale linearly with the sequence length, the sequence length mathematically cancels out when calculating their ratio. Therefore, the compression ratio is determined entirely by the chosen compression parameters, making it strictly independent of sequence length.

**What this tells about where memory savings come from**: The information is not compressed into a hidden state like with an RNN, but rather each individual token's information and the repeated recontextualization during the residual attention blocks are limited to a propogating less information by storing and utilizing a low rank approximation. The information bottleneck is at a token level, not a sequence level.

**Question 3.2 (5 points - Written):** MHLA compresses K and V significantly. What information
might be lost? In what scenarios might this hurt model quality?

**What information might be lost**: Due to the compresion the token infromation, what is lost are variant, or generally unseen high fidelity details. This projection is also linear, so any non-linear or orthogonal feature vectors realtive to the learned distribution will be lost. The lossy compression forces tokens to common representation, so if a token has a unique unseen before or uncommon relationship with other tokens, this will be lost in the compressed representation. 

**In what scenarios might this hurt model quality**: This hurts the model in scenarios where we must capture high-fidelity token information, such as a rare name or a needle in haystack type of problem, algebra or other numerical computations, and copying tasks. This is again due to storing more general token representations instead of highly granular token information, so if the model needs to capture a unique relationship between tokens, this may be lost in the compressed representation.

## Part 4: Putting It All Together (15 points)

See `Transformer.py`.

See `runner.py`.

## Final Questions

### Final Questions

**Question 4.1 (3 points - Written):** Imagine you're deploying a chatbot that needs to handle
conversations of 50,000 tokens. Would you use standard attention or MHLA?
Why? What would be the memory savings?

**Which of standard attention or MHLA you would use**: MHLA

**Why**: While a context window of 50,000 tokens is within the realm of possibility for inference with datacenter scale hardware, on a consumer budget this would be infeasible as the memory requirements could easily be 100s of GBs. Thus, I would still use MHLA as the memory savings would be significant and allow the model to run on consumer hardware, albeit with some potential loss in performance. This would also allow for an increase in throughput and decrease in latency, which are important factors for a chatbot application.

**What the memory savings would be**: In our case, an 8x reduction in memory usage.

**Question 4.2 (3 points - Written):** What's the main trade-off when using a smaller latent dimension
$d_{\text{latent}}$ in MHLA? How would you choose this hyperparameter?

**What the main trade-off when using a smaller latent dimension in MHLA is**: The smaller the latent dimension, the more lossy the compression of the KV cache, which can lead to a significant drop in model performance. However, a smaller latent dimension also leads to greater memory savings, which may be necessary for certain applications or hardware constraints.

**How you would choose the latent dimension**: By defining a maximum allowable memory footprint based on the hardware constraints and then choosing the smallest latent dimension that meets this requirement while still maintaining acceptable model performance. This would likely involve empirical testing and tuning to find the optimal balance between memory savings and model quality.

**Question 4.3 (4 points - Written):** We used Pre-LN (normalize before the block) instead of Post-LN (normalize after the residual). Why is Pre-LN better for deep networks? Hint: think about gradient flow.

**Why Pre-LN is better for deep networks**: Post-LN, or LN(x_l + f(x_l)) means the model must propogate gradients for the residual, particularly earlier layers through the layer normalization operation, which can lead to vanishing gradients and thus training instability. In contrast, Pre-LN, or x_l + f(LN(x_l)) allows the gradients to propogate directly through the residual connection without being attenuated by the normalization operation. The gradient layer has an identity component when the residual is differentiated with respect to the output of the next layer, thus improving gradient flow and enabling more stable training of deeper networks.

Post-LN architectures can be trained with careful initialization and learning rate schedules, but Pre-LN architectures are generally more robust and easier to train, especially as the depth of the network increases. This is why Pre-LN has become the standard in modern transformer architectures.